# Imports

In [ ]:
import pandas as pd
from pathlib import Path
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

# GPU setup
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🚀 MPS GPU!")
else:
    device = torch.device('cpu')
    print("⚠️ CPU")
print(f"Device: {device}")

# Data inladen 

In [ ]:
yawndir = Path('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/yawn-no-yawn')
df_yawn = []

for label_name, label_id in [('no yawn', 0), ('yawn', 1)]:
    label_path = yawndir / label_name
    img_paths = (list(label_path.rglob('*.jpg')) + 
                 list(label_path.rglob('*.png')) + 
                 list(label_path.rglob('*.jpeg')) + 
                 list(label_path.rglob('*.JPG')))
    
    print(f"{label_name}: {len(img_paths)} images")
    
    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            df_yawn.append({'Image': img, 'Label': label_id})

df_yawn = pd.DataFrame(df_yawn)
print(f"✅ Totaal: {df_yawn.shape[0]} images")
print(df_yawn['Label'].value_counts().sort_index())

# Dataset + Dataloaders

In [ ]:
class SimpleYawnDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        img = self.df.iloc[idx]['Image']
        label = self.df.iloc[idx]['Label']
        
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        
        if self.transform:
            img = self.transform(torch.from_numpy(img))
        return img, torch.tensor(label)

train_df, val_df = train_test_split(df_yawn, test_size=0.2, stratify=df_yawn['Label'], random_state=42)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_val = transforms.Compose([transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

train_ds = SimpleYawnDataset(train_df, transform_train)
val_ds = SimpleYawnDataset(val_df, transform_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
print("✅ Dataloaders klaar!")

# Model + Optimizer 

In [ ]:
model_yawn = models.efficientnet_b0(weights='DEFAULT')
model_yawn.classifier[1] = nn.Linear(model_yawn.classifier[1].in_features, 2)
model_yawn = model_yawn.to(device)

optimizer = Adam(model_yawn.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
print("✅ Model geladen!")

# Train/VAl


In [ ]:
def train_epoch(model, loader, opt, crit):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        preds = model(imgs)
        loss = crit(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (preds.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss/total, correct/total

def val_epoch(model, loader, crit):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = crit(preds, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss/total, correct/total

print("✅ Train/val functies klaar!")

# Training loop


In [ ]:
print("🚀 START TRAINING!")
for epoch in range(15):
    tr_loss, tr_acc = train_epoch(model_yawn, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model_yawn, val_loader, criterion)
    print(f"Epoch {epoch+1:2d}: Train {tr_acc:.3f} | Val {val_acc:.3f}")

torch.save(model_yawn.state_dict(), 'yawn_model.pth')
print("✅ YAWN MODEL OPSLAAN!")

# FINE TUNING MODEL met eigen data

In [8]:

from pathlib import Path
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
import torch
import torch.nn as nn
import torchvision.models as models

MY_YAWN_DIR = Path("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/my_yawn_finetune")

extra_rows = []

for label_name, label_id in [("no yawn", 0), ("yawn", 1)]:
    label_path = MY_YAWN_DIR / label_name

    if not label_path.exists():
        print(f"⚠️ Map niet gevonden: {label_path}")
        continue

    img_paths = (
        list(label_path.rglob("*.jpg")) +
        list(label_path.rglob("*.png")) +
        list(label_path.rglob("*.jpeg")) +
        list(label_path.rglob("*.JPG"))
    )

    print(f"{label_name}: {len(img_paths)} extra images")

    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            extra_rows.append({
                "Image": img,
                "Label": label_id
            })

my_yawn_df = pd.DataFrame(extra_rows)
print(f"✅ Eigen yawn images gevonden: {len(my_yawn_df)}")

if len(my_yawn_df) == 0:
    raise ValueError("❌ Geen extra yawn-images gevonden in my_yawn_finetune")

# bestaande df_yawn + eigen data samenvoegen
combined_yawn_df = pd.concat([df_yawn, my_yawn_df], ignore_index=True)
print(f"✅ Totale dataset na toevoegen eigen yawn images: {len(combined_yawn_df)}")
print(combined_yawn_df["Label"].value_counts().sort_index())

# nieuwe split
train_df_ft, val_df_ft = train_test_split(
    combined_yawn_df,
    test_size=0.2,
    stratify=combined_yawn_df["Label"],
    random_state=42
)

# zelfde dataset class en transforms als eerder
train_ds_ft = SimpleYawnDataset(train_df_ft, transform_train)
val_ds_ft = SimpleYawnDataset(val_df_ft, transform_val)

train_loader_ft = DataLoader(train_ds_ft, batch_size=32, shuffle=True, num_workers=0)
val_loader_ft = DataLoader(val_ds_ft, batch_size=32, shuffle=False, num_workers=0)

print(f"✅ Fine-tune Train: {len(train_ds_ft)}, Val: {len(val_ds_ft)}")

# bestaand yawn-model opnieuw opbouwen
finetune_model_yawn = models.efficientnet_b0(weights=None)
finetune_model_yawn.classifier[1] = nn.Linear(finetune_model_yawn.classifier[1].in_features, 2)
finetune_model_yawn = finetune_model_yawn.to(device)

checkpoint_path = "yawn_model.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)
finetune_model_yawn.load_state_dict(checkpoint)

print("✅ Bestaand yawn-model geladen voor fine-tuning")

criterion = CrossEntropyLoss()
optimizer = Adam(finetune_model_yawn.parameters(), lr=1e-4)

EPOCHS = 3

for epoch in range(EPOCHS):
    finetune_model_yawn.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader_ft:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = finetune_model_yawn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(finetune_model_yawn.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    finetune_model_yawn.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader_ft:
            images, labels = images.to(device), labels.to(device)
            outputs = finetune_model_yawn(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

torch.save(finetune_model_yawn.state_dict(), "yawn_model_finetuned.pth")
print("✅ Fine-tuned yawn-model opgeslagen als 'yawn_model_finetuned.pth'")

no yawn: 79 extra images
yawn: 161 extra images
✅ Eigen yawn images gevonden: 240
✅ Totale dataset na toevoegen eigen yawn images: 5359
Label
0    2670
1    2689
Name: count, dtype: int64
✅ Fine-tune Train: 4287, Val: 1072
✅ Bestaand yawn-model geladen voor fine-tuning
Epoch 1/3 | Train Loss: 0.0313 | Train Acc: 0.9853 | Val Loss: 0.0793 | Val Acc: 0.9804
Epoch 2/3 | Train Loss: 0.0219 | Train Acc: 0.9907 | Val Loss: 0.0764 | Val Acc: 0.9804
Epoch 3/3 | Train Loss: 0.0227 | Train Acc: 0.9895 | Val Loss: 0.0802 | Val Acc: 0.9795
✅ Fine-tuned yawn-model opgeslagen als 'yawn_model_finetuned.pth'
